In [1]:
import polars as pl

In [2]:
schema = {
    "mpg": pl.Float64,
    "cylinders": pl.Int64,
    "displacement": pl.Float64,
    "horsepower": pl.Int64,
    "weight": pl.Int64,
    "acceleration": pl.Float64,
    "year": pl.Int64,
    "origin": pl.Int64,
    "name": pl.String,
}

# Loading Data - Eager

In [3]:
df = pl.read_csv(
    "../../datasets/Auto.csv",
    null_values="?",
    # infer_schema_length=10000 # Workaround if inferring fails in Polars
    schema_overrides=schema,  # Can also manually specify
)

In [4]:
print(df.glimpse())

Rows: 397
Columns: 9
$ mpg          <f64> 18.0, 15.0, 18.0, 16.0, 17.0, 15.0, 14.0, 14.0, 14.0, 15.0
$ cylinders    <i64> 8, 8, 8, 8, 8, 8, 8, 8, 8, 8
$ displacement <f64> 307.0, 350.0, 318.0, 304.0, 302.0, 429.0, 454.0, 440.0, 455.0, 390.0
$ horsepower   <i64> 130, 165, 150, 150, 140, 198, 220, 215, 225, 190
$ weight       <i64> 3504, 3693, 3436, 3433, 3449, 4341, 4354, 4312, 4425, 3850
$ acceleration <f64> 12.0, 11.5, 11.0, 12.0, 10.5, 10.0, 9.0, 8.5, 10.0, 8.5
$ year         <i64> 70, 70, 70, 70, 70, 70, 70, 70, 70, 70
$ origin       <i64> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1
$ name         <str> 'chevrolet chevelle malibu', 'buick skylark 320', 'plymouth satellite', 'amc rebel sst', 'ford torino', 'ford galaxie 500', 'chevrolet impala', 'plymouth fury iii', 'pontiac catalina', 'amc ambassador dpl'

None


# Loading data - Lazy

In [5]:
lazy_df = pl.scan_csv(
    "../../datasets/Auto.csv",
    null_values="?",
    # infer_schema_length=10000 # Workaround if inferring fails in Polars
    schema_overrides=schema,  # Can also manually specify
)

# Horsepower - Datacleaning

In [6]:
q = lazy_df.select(pl.col("horsepower").unique())

q.collect()

horsepower
i64
null
46
48
49
52
…
210
215
220


In [7]:
q = lazy_df.drop_nulls().select(pl.col("*"))
df = q.collect()

In [35]:
df.unique(subset=["horsepower"], maintain_order=True).select(pl.col("horsepower"))

horsepower
i64
130
165
150
140
198
…
84
64
74


In [12]:
summed_horsepower = pl.col("horsepower").sum()
df.select(summed_horsepower=summed_horsepower)

summed_horsepower
i64
40952


# Selecting data

In [16]:
year_80 = pl.col("year") == 80
df.filter(year_80)

mpg,cylinders,displacement,horsepower,weight,acceleration,year,origin,name
f64,i64,f64,i64,i64,f64,i64,i64,str
41.5,4,98.0,76,2144,14.7,80,2,"""vw rabbit"""
38.1,4,89.0,60,1968,18.8,80,3,"""toyota corolla tercel"""
32.1,4,98.0,70,2120,15.5,80,1,"""chevrolet chevette"""
37.2,4,86.0,65,2019,16.4,80,3,"""datsun 310"""
28.0,4,151.0,90,2678,16.5,80,1,"""chevrolet citation"""
…,…,…,…,…,…,…,…,…
29.8,4,89.0,62,1845,15.3,80,2,"""vokswagen rabbit"""
32.7,6,168.0,132,2910,11.4,80,3,"""datsun 280-zx"""
23.7,3,70.0,100,2420,12.5,80,3,"""mazda rx-7 gs"""


In [18]:
df.filter(pl.col("name").is_in(["amc rebel sst", "ford torino"]))

mpg,cylinders,displacement,horsepower,weight,acceleration,year,origin,name
f64,i64,f64,i64,i64,f64,i64,i64,str
16.0,8,304.0,150,3433,12.0,70,1,"""amc rebel sst"""
17.0,8,302.0,140,3449,10.5,70,1,"""ford torino"""


In [24]:
df[2:4]

mpg,cylinders,displacement,horsepower,weight,acceleration,year,origin,name
f64,i64,f64,i64,i64,f64,i64,i64,str
18.0,8,318.0,150,3436,11.0,70,1,"""plymouth satellite"""
16.0,8,304.0,150,3433,12.0,70,1,"""amc rebel sst"""


In [25]:
columns = [pl.col("mpg"), pl.col("displacement"), pl.col("horsepower")]
df.select(columns)

mpg,displacement,horsepower
f64,f64,i64
18.0,307.0,130
15.0,350.0,165
18.0,318.0,150
16.0,304.0,150
17.0,302.0,140
…,…,…
27.0,140.0,86
44.0,97.0,52
32.0,135.0,84


In [26]:
df[2:4].select(columns)

mpg,displacement,horsepower
f64,f64,i64
18.0,318.0,150
16.0,304.0,150


In [28]:
(
    df.select(pl.col("mpg"), pl.col("origin"), pl.col("name")).filter(
        pl.col("name") == "ford galaxie 500"
    )
)

mpg,origin,name
f64,i64,str
15.0,1,"""ford galaxie 500"""
14.0,1,"""ford galaxie 500"""
14.0,1,"""ford galaxie 500"""


In [30]:
(
    df.select(pl.col("weight"), pl.col("origin"), pl.col("year")).filter(
        pl.col("year") > 80
    )
)

weight,origin,year
i64,i64,i64
2490,1,81
2635,1,81
2620,1,81
2725,1,81
2385,1,81
…,…,…
2790,1,82
2130,2,82
2295,1,82


# Advanced selection

In [34]:
(
    df.select(
        pl.col("name"), pl.col("displacement"), pl.col("weight"), pl.col("origin")
    ).filter(
        pl.col("displacement") > 80, pl.col("name").str.contains_any(["ford", "datsun"])
    )
)

name,displacement,weight,origin
str,f64,i64,i64
"""ford torino""",302.0,3449,1
"""ford galaxie 500""",429.0,4341,1
"""ford maverick""",200.0,2587,1
"""datsun pl510""",97.0,2130,3
"""ford f250""",360.0,4615,1
…,…,…,…
"""ford fairmont futura""",140.0,2865,1
"""datsun 310 gx""",91.0,1995,3
"""ford granada l""",232.0,2835,1
